# Solutions · 15 · Drying and evaporation

Worked solutions to the exercises of [notebook 15](../notebooks/15_drying_and_evaporation.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import datasets
from engsep import drying_evaporation as de
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
d = pd.read_csv(datasets.path("drying_curve.csv"), comment="#")
an = de.drying_curve_analysis(d.t_s, d.X_dry_basis)
X_c, X_eq, R_c = an["X_critical"], 0.06, an["R_c"]

## Exercise 1

The cake is dried with air at 80 degC instead of 60 (same humidity): estimate the new constant rate from the wet-bulb driving force (R_c proportional to T - T_wb) and the new drying time to X = 0.10. What about the falling-rate period - does hotter air help there?

In [3]:
H = 0.01
T_wb60, T_wb80 = de.wet_bulb_temperature(60.0, H), de.wet_bulb_temperature(80.0, H)
R_c80 = R_c * (80.0 - T_wb80) / (60.0 - T_wb60)
t60 = de.drying_time(0.9, 0.10, X_c, X_eq, R_c, 1.0)["t_total"]
t80 = de.drying_time(0.9, 0.10, X_c, X_eq, R_c80, 1.0)["t_total"]
print(f"wet-bulb {T_wb60:.1f} degC at 60 degC, {T_wb80:.1f} at 80: constant rate x{R_c80/R_c:.2f}; drying time {t60/60:.0f} -> {t80/60:.0f} min")
print("the falling-rate period is shortened in the same proportion here because the model scales it with R_c; in reality internal diffusion depends on the solid's temperature and structure, and hotter air helps less than proportionally")

wet-bulb 27.7 degC at 60 degC, 31.9 at 80: constant rate x1.49; drying time 199 -> 134 min
the falling-rate period is shortened in the same proportion here because the model scales it with R_c; in reality internal diffusion depends on the solid's temperature and structure, and hotter air helps less than proportionally


Hotter air raises the wet-bulb driving force by about 40 % and shortens the constant-rate period
accordingly; the simple model shortens the falling-rate period too, but real falling-rate drying is
limited by moisture movement inside the solid, which air temperature affects only through the solid's
temperature. Hot air is the lever for the surface, not the interior.

## Exercise 2

Backward feed: in a triple effect, feed the cold solution to the last (coldest) effect and pump it towards the first. Argue, with the single-effect function applied effect by effect, why backward feed gives a better economy for a cold feed and a viscous concentrate.

In [4]:
# backward feed by hand: the cold feed enters the coldest effect (3), is heated there by the vapour of effect 2, and so on
F, x_f, x_p = 2.0, 0.10, 0.40
V_total = F * (1 - x_f / x_p)
T = [107.0, 90.0, 60.0]                                                  # boiling temperatures of effects 1, 2, 3 (as in the forward-feed table)
# forward feed: cold feed to effect 1 at 107 degC
Q_fwd_1 = F * 4000 * (T[0] - 25.0) + V_total / 3 * de.latent_heat(T[0])
# backward feed: cold feed to effect 3 at 60 degC, then pumped hotter: each effect heats its liquor by the step to its own temperature
Q_bwd_3 = F * 4000 * (T[2] - 25.0) + V_total / 3 * de.latent_heat(T[2])
Q_bwd_2 = (F - V_total / 3) * 4000 * (T[1] - T[2]) + V_total / 3 * de.latent_heat(T[1])
Q_bwd_1 = (F - 2 * V_total / 3) * 4000 * (T[0] - T[1]) + V_total / 3 * de.latent_heat(T[0])
print(f"forward feed: the first effect must supply {Q_fwd_1/1e6:.2f} MW (of which {F*4000*(T[0]-25)/1e6:.2f} MW heats the cold feed) with live steam")
print(f"backward feed: live steam supplies only effect 1's {Q_bwd_1/1e6:.2f} MW; the feed's sensible heating ({F*4000*(T[2]-25)/1e6:.2f} MW) is done in effect 3 by cheap low-pressure vapour")

forward feed: the first effect must supply 1.78 MW (of which 0.66 MW heats the cold feed) with live steam
backward feed: live steam supplies only effect 1's 1.19 MW; the feed's sensible heating (0.28 MW) is done in effect 3 by cheap low-pressure vapour


With forward feed the cold feed is heated in the first effect by live steam, the most expensive heat in the
train; with backward feed it enters the last effect and is heated by the last effect's vapour, which would
otherwise go to the condenser. Backward feed also puts the viscous concentrate in the hottest effect, where
it is least viscous. The price is pumps between effects and the concentrate leaving hot.

## Exercise 3

**Implement it yourself:** write the single-effect balance yourself (vapour from the concentration ratio, duty from sensible + latent heat, steam from the latent heat at the steam temperature) and reproduce `single_effect`.

In [5]:
F, x_f, x_p, T_f, T_b, T_s, U, bpr = 2.0, 0.10, 0.40, 25.0, 100.0, 120.0, 2000.0, 3.0
V = F * (1 - x_f / x_p)
T_v = T_b + bpr
Q = F * 4000.0 * (T_v - T_f) + V * de.latent_heat(T_v)
steam = Q / de.latent_heat(T_s)
lib = de.single_effect(F, x_f, x_p, T_f, T_b, T_s, U, bpr=bpr)
print(f"by hand: V = {V:.3f}, Q = {Q/1e6:.3f} MW, steam = {steam:.3f} kg/s, area = {Q/(U*(T_s - T_v)):.1f} m2; library: {lib['vapour_kg_s']:.3f}, {lib['Q_W']/1e6:.3f}, {lib['steam_kg_s']:.3f}, {lib['area_m2']:.1f}")

by hand: V = 1.500, Q = 4.011 MW, steam = 1.808 kg/s, area = 118.0 m2; library: 1.500, 4.011, 1.808, 118.0


The single-effect balance is four lines of arithmetic, reproduced exactly.